# Phase 5A: Monitoring & Evaluation Dashboard

Real-time monitoring of API performance, costs, and quality metrics.

In [ ]:
import sys
sys.path.insert(0, '/Users/prajwalchambenandeeshappa/Github_Repos/Stocks_Earnings_Intelligence_Agent-Text2SQL/learning')

import psycopg2
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
conn = psycopg2.connect(host='localhost', port=5432, database='financial_data', user='postgres', password='postgres')
sns.set_style('darkgrid')
print('✅ Connected to PostgreSQL')

---
## Performance Metrics (Last 24 Hours)

In [ ]:
cursor = conn.cursor()
cursor.execute('''
    SELECT COUNT(*) as total, AVG(response_time_ms) as avg_latency, MAX(response_time_ms) as max_latency,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY response_time_ms) as p95,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY response_time_ms) as p99
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '24 hours'
''')

total, avg_lat, max_lat, p95, p99 = cursor.fetchone()

print('='*60)
print('PERFORMANCE METRICS (Last 24 Hours)')
print('='*60)
print(f'Total Queries: {total or 0}')
print(f'Avg Latency: {avg_lat or 0:.0f}ms')
print(f'Max Latency: {max_lat or 0:.0f}ms')
print(f'P95 Latency: {p95 or 0:.0f}ms')
print(f'P99 Latency: {p99 or 0:.0f}ms')
print('='*60)

In [ ]:
# Latency trend
cursor.execute('''
    SELECT DATE_TRUNC('hour', timestamp) as hour, AVG(response_time_ms) as avg_latency, COUNT(*) as count
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '24 hours'
    GROUP BY DATE_TRUNC('hour', timestamp) ORDER BY hour
''')

data = cursor.fetchall()
df_latency = pd.DataFrame(data, columns=['Hour', 'Avg_Latency', 'Count'])

if not df_latency.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.plot(df_latency['Hour'], df_latency['Avg_Latency'], marker='o', linewidth=2)
    ax1.set_title('Latency Trend (24h)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Hour')
    ax1.set_ylabel('Latency (ms)')
    ax1.grid(True, alpha=0.3)
    
    ax2.bar(df_latency['Hour'], df_latency['Count'], color='steelblue', alpha=0.7)
    ax2.set_title('Request Volume (24h)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Hour')
    ax2.set_ylabel('Number of Requests')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print('No data available')

---
## Cost Analysis

In [ ]:
cursor.execute('''
    SELECT SUM(cost_usd) as total_cost, COUNT(*) as requests, AVG(cost_usd) as avg_cost
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '30 days'
''')

total_cost, requests, avg_cost = cursor.fetchone()

print('='*60)
print('COST ANALYSIS (Last 30 Days)')
print('='*60)
print(f'Total Cost: ${total_cost or 0:.2f}')
print(f'Total Requests: {requests or 0}')
print(f'Avg Cost/Request: ${avg_cost or 0:.4f}')
print(f'Daily Average: ${(total_cost or 0) / 30:.2f}')
print('='*60)

In [ ]:
# Daily cost trend
cursor.execute('''
    SELECT DATE(timestamp) as date, SUM(cost_usd) as daily_cost, COUNT(*) as requests
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '30 days'
    GROUP BY DATE(timestamp) ORDER BY date
''')

data = cursor.fetchall()
df_cost = pd.DataFrame(data, columns=['Date', 'Cost', 'Requests'])

if not df_cost.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.bar(df_cost['Date'], df_cost['Cost'], color='green', alpha=0.7)
    ax1.set_title('Daily API Costs (30d)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Cost (USD)')
    ax1.grid(True, alpha=0.3, axis='y')
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    
    ax2.plot(df_cost['Date'], df_cost['Requests'], marker='o', color='orange', linewidth=2)
    ax2.set_title('Daily Request Volume (30d)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Number of Requests')
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()

---
## Quality Metrics

In [ ]:
cursor.execute('''
    SELECT AVG(retrieval_precision) as avg_precision, AVG(answer_quality) as avg_quality,
    COUNT(CASE WHEN answer_quality >= 4 THEN 1 END)::float / COUNT(*) * 100 as good_pct
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '7 days'
''')

avg_prec, avg_qual, good_pct = cursor.fetchone()

print('='*60)
print('QUALITY METRICS (Last 7 Days)')
print('='*60)
print(f'Avg Retrieval Precision: {(avg_prec or 0)*100:.1f}%')
print(f'Avg Answer Quality: {avg_qual or 0:.2f}/5')
print(f'Good Quality Rate (≥4): {good_pct or 0:.1f}%')
print('='*60)

---
## User Feedback & Ratings

In [ ]:
cursor.execute('''
    SELECT user_feedback, COUNT(*) as count FROM evaluation_logs
    WHERE user_feedback IS NOT NULL GROUP BY user_feedback ORDER BY user_feedback DESC
''')

ratings = cursor.fetchall()

if ratings:
    rating_dict = {r[0]: r[1] for r in ratings}
    avg_rating = sum(r[0]*r[1] for r in ratings) / sum(r[1] for r in ratings)
    
    print('='*60)
    print('USER FEEDBACK SUMMARY')
    print('='*60)
    print(f'Average Rating: {avg_rating:.2f}/5')
    print(f'Total Feedback: {sum(r[1] for r in ratings)}')
    print('\nBreakdown:')
    for rating, count in sorted(rating_dict.items(), reverse=True):
        print(f'  {rating}⭐: {count} ({count/sum(r[1] for r in ratings)*100:.1f}%)')
    print('='*60)
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    ratings_list = sorted(rating_dict.keys())
    counts = [rating_dict[r] for r in ratings_list]
    
    bars = ax.bar([f'{r}⭐' for r in ratings_list], counts, color=['red', 'orange', 'yellow', 'lightgreen', 'green'])
    ax.set_title('User Rating Distribution', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Ratings')
    ax.set_xlabel('Rating')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print('No feedback data available')

---
## System Health Summary

In [ ]:
cursor.execute('''
    SELECT 
    COUNT(*) as total_queries,
    COUNT(CASE WHEN response_time_ms < 1000 THEN 1 END)::float / COUNT(*) * 100 as fast_pct,
    COUNT(CASE WHEN response_time_ms > 3000 THEN 1 END)::float / COUNT(*) * 100 as slow_pct,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY response_time_ms) as median_latency
    FROM evaluation_logs WHERE timestamp > NOW() - INTERVAL '24 hours'
''')

total, fast, slow, median = cursor.fetchone()

print('='*60)
print('SYSTEM HEALTH STATUS')
print('='*60)
print(f'\n📊 Response Time Distribution (24h):')
print(f'  Fast (<1s): {fast or 0:.1f}%')
print(f'  Normal (1-3s): {100 - (fast or 0) - (slow or 0):.1f}%')
print(f'  Slow (>3s): {slow or 0:.1f}%')
print(f'  Median Latency: {median or 0:.0f}ms')

# Health status
if (slow or 0) < 1:
    status = '🟢 HEALTHY'
elif (slow or 0) < 5:
    status = '🟡 WARNING'
else:
    status = '🔴 CRITICAL'

print(f'\nOverall Status: {status}')
print('='*60)